# EP04. Agent Harness를 활용한 성능 평가

## 학습 목표

1. **Agent Harness**의 개념을 이해하고 직접 구현한다
2. LangGraph 에이전트에 **자동화된 테스트 파이프라인**을 연결한다
3. **Langfuse Dataset**으로 테스트 케이스를 중앙 관리한다
4. **Langfuse Experiment**로 모델 A vs B 성능을 데이터로 비교한다
5. Pass@1, F1, 비용, 지연시간 **4대 메트릭**을 측정한다

---

> **AI 가이드**: 이 노트북은 순서대로 실행하도록 설계되었습니다.  
> 섹션 1에서 패키지를 설치하고, 섹션 2에서 API 키를 설정한 후 진행하세요.  
> Langfuse 계정이 없다면 [cloud.langfuse.com](https://cloud.langfuse.com)에서 무료로 생성하세요.  
> 각 코드 셀을 실행하면 중간 결과가 즉시 확인됩니다.


## 섹션 1. 환경 설정

In [ ]:
# 필요한 패키지 설치
!uv pip install langchain langgraph langchain-anthropic langfuse python-dotenv

In [ ]:
# 설치 확인
import langchain, langgraph, langfuse
print(f'langchain: {langchain.__version__}')
print(f'langgraph: {langgraph.__version__}')
print(f'langfuse:  {langfuse.__version__}')

## 섹션 2. 라이브러리 로드 + Langfuse 초기화

In [ ]:
import os
import time
import json
from dotenv import load_dotenv

load_dotenv()

# API 키 설정 (환경 변수 또는 직접 입력)
# os.environ['ANTHROPIC_API_KEY']    = 'sk-ant-...'
# os.environ['LANGFUSE_PUBLIC_KEY']  = 'pk-lf-...'
# os.environ['LANGFUSE_SECRET_KEY']  = 'sk-lf-...'
# os.environ['LANGFUSE_HOST']        = 'https://cloud.langfuse.com'

from langfuse import Langfuse

langfuse = Langfuse()
langfuse.auth_check()
print('Langfuse 연결 성공!')

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langfuse.langchain import CallbackHandler

print('라이브러리 로드 완료')

## 섹션 3. 평가용 테스트 케이스 10개 정의

도구 없이 또는 간단한 계산기 도구만으로 풀 수 있는 테스트 케이스입니다.  
각 케이스는 `id`, `input`, `expected`, `category`, `level` 필드를 포함합니다.


In [ ]:
test_cases = [
    # Level 1: 직접 계산 또는 상식
    {'id': 'TC-001', 'input': '123 곱하기 456은 얼마인가요?', 'expected': '56088',
     'category': 'math', 'level': 1},
    {'id': 'TC-002', 'input': '1024 나누기 32는 얼마인가요?', 'expected': '32',
     'category': 'math', 'level': 1},
    {'id': 'TC-003', 'input': '대한민국의 수도는 어디인가요?', 'expected': '서울',
     'category': 'factual', 'level': 1},
    {'id': 'TC-004', 'input': '2024년은 윤년인가요?', 'expected': '예',
     'category': 'calendar', 'level': 1},
    # Level 2: 다단계 계산
    {'id': 'TC-005', 'input': '피보나치 수열에서 10번째 숫자는 무엇인가요?', 'expected': '55',
     'category': 'math', 'level': 2},
    {'id': 'TC-006', 'input': '100 이하의 소수는 몇 개인가요?', 'expected': '25',
     'category': 'math', 'level': 2},
    {'id': 'TC-007', 'input': '5 마일은 몇 킬로미터인가요? (1마일=1.60934km)', 'expected': '8.0467',
     'category': 'unit_conversion', 'level': 2},
    {'id': 'TC-008', 'input': '2025년 1월 1일이 수요일이라면 2025년 3월 1일은 무슨 요일인가요?',
     'expected': '토요일', 'category': 'calendar', 'level': 2},
    # Level 3: 복합 추론
    {'id': 'TC-009', 'input': 'A > B, B > C, C > D 이면 A와 D의 관계는?',
     'expected': 'A > D', 'category': 'logic', 'level': 3},
    {'id': 'TC-010', 'input': '어떤 수를 3으로 나누면 나머지가 2이고, 5로 나누면 나머지가 1입니다. 가장 작은 양의 정수는?',
     'expected': '11', 'category': 'math', 'level': 3},
]

print(f'테스트 케이스 총 {len(test_cases)}개 정의 완료')
for tc in test_cases:
    print(f"  {tc['id']} | Level {tc['level']} | {tc['category']} | {tc['input'][:45]}")

## 섹션 4. LangGraph 에이전트 구축

**[Agent Harness 아키텍처]**
```mermaid
flowchart LR
    A[/"📁 TestSuite
(테스트 케이스)"\]:::req --> B("⚙️ Runner
(일괄 실행 엔진)"):::runner
    B <--> C{"🤖 Agent
(LangGraph)"}:::agent
    B --> D("✅ Evaluator
(응답 품질 채점)"):::eval
    D --> E("📊 Reporter
(결과 집계/시각화)"):::report
    E --> F[("☁️ Langfuse
(실험 기록/비교)")]:::cloud
    classDef req fill:#e3f2fd,stroke:#1e88e5,stroke-width:2px,color:#000
    classDef runner fill:#bbdefb,stroke:#1976d2,stroke-width:2px,color:#000
    classDef agent fill:#ffecb3,stroke:#ffb300,stroke-width:2px,rx:10px,ry:10px,color:#000
    classDef eval fill:#c8e6c9,stroke:#43a047,stroke-width:2px,color:#000
    classDef report fill:#e1bee7,stroke:#8e24aa,stroke-width:2px,color:#000
    classDef cloud fill:#b2dfdb,stroke:#00897b,stroke-width:2px,rx:10px,ry:10px,color:#000
```

계산기 도구를 포함한 ReAct 에이전트를 만듭니다.


In [ ]:
import math

@tool
def calculator(expression: str) -> str:
    """수학 표현식을 계산합니다. Python 수식 형태로 입력하세요.
    예: '123 * 456', 'math.sqrt(144)', 'sum(range(1, 11))'
    """
    try:
        allowed = {'math': math, 'sum': sum, 'range': range, 'abs': abs, 'round': round}
        result = eval(expression, {'__builtins__': {}}, allowed)
        return str(result)
    except Exception as e:
        return f'계산 오류: {e}'


def build_agent(model_name: str):
    """지정된 모델로 ReAct 에이전트를 생성합니다."""
    model = ChatAnthropic(model=model_name, temperature=0)
    return create_react_agent(model, tools=[calculator])


agent_haiku = build_agent('claude-haiku-4-5')
print('에이전트 구축 완료: claude-haiku-4-5')

# 빠른 동작 테스트
test_resp = agent_haiku.invoke({'messages': [('user', '2 더하기 3은?')]})
print('테스트 응답:', test_resp['messages'][-1].content)

## 섹션 5. Harness 실행 함수: run_test_suite

`run_test_suite(agent, test_cases)` 함수는 모든 테스트 케이스를 실행하고  
결과를 딕셔너리 리스트로 반환합니다.


In [ ]:
def evaluate_answer(answer: str, expected: str) -> float:
    """응답과 정답을 비교하여 0.0~1.0 점수를 반환합니다."""
    a = answer.strip().replace(',', '').lower()
    e = expected.strip().replace(',', '').lower()
    if e in a:
        return 1.0
    try:
        a_num = float(a.split()[0])
        e_num = float(e.split()[0])
        if abs(a_num - e_num) / (abs(e_num) + 1e-9) < 0.02:
            return 1.0
    except Exception:
        pass
    return 0.0


def run_test_suite(agent, test_cases: list, experiment_name: str = 'default') -> list:
    """에이전트를 테스트 케이스로 일괄 실행합니다. 결과 딕셔너리 리스트를 반환합니다."""
    results = []
    for tc in test_cases:
        print(f"  [{tc['id']}] 실행 중...", end=' ')
        start = time.time()
        try:
            response = agent.invoke(
                {'messages': [('user', tc['input'])]},
                config={'recursion_limit': 15}
            )
            answer = response['messages'][-1].content
            latency = time.time() - start
            score = evaluate_answer(answer, tc['expected'])
            status = 'PASS' if score >= 0.5 else 'FAIL'
        except Exception as exc:
            answer = f'ERROR: {exc}'
            latency = time.time() - start
            score, status = 0.0, 'ERROR'

        results.append({
            'id': tc['id'], 'category': tc['category'], 'level': tc['level'],
            'input': tc['input'], 'expected': tc['expected'],
            'answer': answer[:120], 'score': score,
            'latency': round(latency, 2), 'status': status,
            'experiment': experiment_name,
        })
        print(f'{status} ({latency:.1f}s)')
    return results

print('Harness 함수 정의 완료')

## 섹션 6. 성공률 · 응답 시간 측정

In [ ]:
import pandas as pd

print('=== Harness 실행: claude-haiku-4-5 ===\n')
raw_haiku = run_test_suite(agent_haiku, test_cases, 'haiku-baseline')
df_haiku = pd.DataFrame(raw_haiku)

pass_at_1   = df_haiku['score'].mean()
avg_latency = df_haiku['latency'].mean()

print(f'\n--- 결과 요약 ---')
print(f'Pass@1:        {pass_at_1:.2%}')
print(f'평균 응답 시간: {avg_latency:.2f}초')
print(f"통과 케이스:    {df_haiku[df_haiku['status']=='PASS'].shape[0]} / {len(test_cases)}개")
print()
print('레벨별 성공률:')
print(df_haiku.groupby('level')['score'].mean().to_string())
print()
display(df_haiku[['id', 'level', 'category', 'expected', 'status', 'score', 'latency']])

## 섹션 7. Langfuse Dataset 생성 및 테스트 케이스 업로드

Langfuse Dataset으로 테스트 세트를 중앙 관리합니다.  
팀 전체가 동일한 테스트 케이스를 공유하고, 실험 결과와 함께 추적할 수 있습니다.


In [ ]:
DATASET_NAME = 'ep04-agent-harness-v1'

try:
    dataset = langfuse.get_dataset(DATASET_NAME)
    print(f'기존 데이터셋 로드: {DATASET_NAME}')
except Exception:
    dataset = langfuse.create_dataset(
        name=DATASET_NAME,
        description='EP04 Agent Harness 평가 데이터셋 v1.0'
    )
    print(f'새 데이터셋 생성: {DATASET_NAME}')

for tc in test_cases:
    langfuse.create_dataset_item(
        dataset_name=DATASET_NAME,
        input={'question': tc['input']},
        expected_output={'answer': tc['expected']},
        metadata={'id': tc['id'], 'level': tc['level'], 'category': tc['category']}
    )

langfuse.flush()
print(f'{len(test_cases)}개 테스트 케이스 업로드 완료')
print(f'Langfuse → Datasets → {DATASET_NAME}')

## 섹션 8. Langfuse Experiment 실행

`run_experiment` 함수는 Langfuse Dataset을 불러와 에이전트를 실행하고  
각 결과를 Langfuse score API로 자동 기록합니다.


In [ ]:
def run_experiment(agent, dataset_name: str, run_name: str, model_name: str = 'unknown') -> pd.DataFrame:
    """Langfuse Dataset 기반 실험을 실행하고 결과를 DataFrame으로 반환합니다."""
    dataset = langfuse.get_dataset(dataset_name)
    results = []

    for item in dataset.items:
        question = item.input['question']
        expected = item.expected_output['answer']
        tc_id    = item.metadata.get('id', '?')
        level    = item.metadata.get('level', 0)
        category = item.metadata.get('category', '?')

        print(f'  [{tc_id}] 실행 중...', end=' ')
        start = time.time()

        with item.observe(run_name=run_name) as trace:
            try:
                langfuse_cb = CallbackHandler()
                response = agent.invoke(
                    {'messages': [('user', question)]},
                    config={'recursion_limit': 15, 'callbacks': [langfuse_cb]}
                )
                answer  = response['messages'][-1].content
                latency = time.time() - start
                score   = evaluate_answer(answer, expected)
                status  = 'PASS' if score >= 0.5 else 'FAIL'
                trace.score(name='correctness', value=score,
                             comment=f'expected={expected}, got={answer[:40]}')
                trace.score(name='latency_seconds', value=round(latency, 2))
            except Exception as exc:
                answer  = f'ERROR: {exc}'
                latency = time.time() - start
                score, status = 0.0, 'ERROR'

        results.append({
            'id': tc_id, 'level': level, 'category': category,
            'question': question[:60], 'expected': expected,
            'answer': answer[:80], 'score': score,
            'latency': round(latency, 2), 'status': status,
            'model': model_name, 'run': run_name,
        })
        print(f'{status} (score={score:.1f}, {latency:.1f}s)')

    langfuse.flush()
    return pd.DataFrame(results)

print('run_experiment 함수 정의 완료')

## 섹션 9. 모델 A(Haiku) vs 모델 B(Sonnet) 비교

동일한 Langfuse Dataset으로 두 모델을 실행하여 성능을 비교합니다.  
결과는 Langfuse 대시보드에서도 나란히 확인할 수 있습니다.


In [ ]:
print('=== 실험 A: claude-haiku-4-5 ===\n')
df_exp_haiku = run_experiment(
    agent=build_agent('claude-haiku-4-5'),
    dataset_name=DATASET_NAME,
    run_name='experiment-haiku',
    model_name='claude-haiku-4-5'
)
print()

print('=== 실험 B: claude-3-5-sonnet-20241022 ===\n')
df_exp_sonnet = run_experiment(
    agent=build_agent('claude-3-5-sonnet-20241022'),
    dataset_name=DATASET_NAME,
    run_name='experiment-sonnet',
    model_name='claude-3-5-sonnet-20241022'
)
print('\n두 실험 완료!')

## 섹션 10. 결과 DataFrame + 리더보드 시각화

In [ ]:
def summarize(df: pd.DataFrame) -> dict:
    return {
        'model':        df['model'].iloc[0],
        'pass_at_1':    round(df['score'].mean(), 4),
        'avg_latency':  round(df['latency'].mean(), 2),
        'pass_count':   int(df['status'].eq('PASS').sum()),
        'total':        len(df),
        'level1':       round(df[df['level']==1]['score'].mean(), 3),
        'level2':       round(df[df['level']==2]['score'].mean(), 3),
        'level3':       round(df[df['level']==3]['score'].mean(), 3),
    }

leaderboard = pd.DataFrame([
    summarize(df_exp_sonnet),
    summarize(df_exp_haiku),
]).sort_values('pass_at_1', ascending=False).reset_index(drop=True)
leaderboard.index = ['1위', '2위']

print('=' * 55)
print('         모델 성능 리더보드')
print('=' * 55)
display(leaderboard)

# 레벨별 비교
combined = pd.concat([df_exp_haiku, df_exp_sonnet])
print('\n--- 레벨별 성공률 비교 ---')
display(combined.groupby(['model', 'level'])['score'].mean().unstack())

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib
    matplotlib.rcParams['font.family'] = 'AppleGothic'

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    s_haiku  = summarize(df_exp_haiku)
    s_sonnet = summarize(df_exp_sonnet)
    models   = ['haiku-4-5', 'sonnet-20241022']
    colors   = ['#5b9bd5', '#ed7d31']

    axes[0].bar(models, [s_haiku['pass_at_1'], s_sonnet['pass_at_1']], color=colors)
    axes[0].set_title('Pass@1 비교'); axes[0].set_ylim(0, 1)

    axes[1].bar(models, [s_haiku['avg_latency'], s_sonnet['avg_latency']], color=colors)
    axes[1].set_title('평균 응답 시간 (초)')

    lvl_data = combined.groupby(['model', 'level'])['score'].mean().unstack().T
    x, w = range(len(lvl_data)), 0.35
    axes[2].bar([i - w/2 for i in x], lvl_data.iloc[:, 0], w, label=models[0], color=colors[0])
    axes[2].bar([i + w/2 for i in x], lvl_data.iloc[:, 1], w, label=models[1], color=colors[1])
    axes[2].set_title('레벨별 성공률')
    axes[2].set_xticks(x); axes[2].set_xticklabels([f'Level {i+1}' for i in x])
    axes[2].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig('ep04_leaderboard.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('그래프 저장: ep04_leaderboard.png')
except ImportError:
    print('matplotlib 미설치 — 텍스트 결과만 표시')

## 섹션 11. 자동화 회귀 테스트 패턴

**[자동화 회귀 테스트(CI/CD) 파이프라인]**
```mermaid
flowchart TD
    A("💻 코드/프롬프트 변경"):::code --> B("🚀 CI/CD 파이프라인 트리거"):::cicd
    B --> C[/"��️ Langfuse Dataset 로드"\]:::data
    C --> D("🏃 에이전트 일괄 실행 (Runner)"):::run
    D --> E("⚖️ Evaluator 자동 채점"):::score
    E --> F{"🧐 평가 기준 만족?
(Pass@1 >= 0.90)"}:::decision
    F -->|Yes| G("🚀 프로덕션 배포 승인"):::pass
    G --> I[("📈 Langfuse Experiment 기록")]:::log
    F -->|No| H("🛑 배포 차단 및 알림"):::fail
    H --> J("🧑‍💻 개발자 리뷰 및 디버깅"):::review
    classDef code fill:#eceff1,stroke:#90a4ae,stroke-width:2px,color:#000
    classDef cicd fill:#bbdefb,stroke:#1e88e5,stroke-width:2px,color:#000
    classDef data fill:#fff3e0,stroke:#fb8c00,stroke-width:2px,color:#000
    classDef run fill:#e1bee7,stroke:#8e24aa,stroke-width:2px,color:#000
    classDef score fill:#b2ebf2,stroke:#00acc1,stroke-width:2px,color:#000
    classDef decision fill:#ffcc80,stroke:#f57c00,stroke-width:2px,font-weight:bold,color:#000
    classDef pass fill:#c8e6c9,stroke:#43a047,stroke-width:2px,font-weight:bold,color:#000
    classDef fail fill:#ffcdd2,stroke:#e53935,stroke-width:2px,font-weight:bold,color:#000
    classDef log fill:#dcedc8,stroke:#7cb342,stroke-width:2px,color:#000
    classDef review fill:#f8bbd0,stroke:#d81b60,stroke-width:2px,color:#000
```

새 에이전트 버전 배포 전 성능이 기준선 아래로 떨어지지 않았는지 자동 검증합니다.  
CI/CD 파이프라인에서 이 함수가 `False`를 반환하면 배포를 차단합니다.


In [ ]:
BASELINE_PASS_AT_1 = 0.80   # Pass@1 기준선
BASELINE_LATENCY   = 10.0  # 응답 시간 기준선(초)


def regression_check(df: pd.DataFrame,
                      baseline_pass: float = BASELINE_PASS_AT_1,
                      baseline_latency: float = BASELINE_LATENCY) -> bool:
    """회귀 테스트: 기준선 대비 성능 저하 여부 확인. True=통과, False=실패."""
    current_pass    = df['score'].mean()
    current_latency = df['latency'].mean()

    print('\n--- 회귀 테스트 결과 ---')
    print(f'Pass@1:  {current_pass:.2%}  (기준선: {baseline_pass:.2%})')
    print(f'Latency: {current_latency:.2f}s (기준선: {baseline_latency:.1f}s)')

    pass_ok    = current_pass    >= baseline_pass
    latency_ok = current_latency <= baseline_latency

    if pass_ok and latency_ok:
        print('결과: PASS — 배포 승인')
        return True

    reasons = []
    if not pass_ok:
        reasons.append(f'Pass@1 기준선 미달 ({current_pass:.2%} < {baseline_pass:.2%})')
    if not latency_ok:
        reasons.append(f'Latency 기준선 초과 ({current_latency:.2f}s > {baseline_latency:.1f}s)')
    print('결과: FAIL — 배포 차단')
    for r in reasons:
        print(f'  - {r}')
    return False


print('=== Haiku 회귀 테스트 ===')
haiku_ok = regression_check(df_exp_haiku)

print('\n=== Sonnet 회귀 테스트 ===')
sonnet_ok = regression_check(df_exp_sonnet)

## Exercise

### Exercise 1: 5개 테스트 케이스로 Harness 구축

아래 5개 카테고리에서 각각 1개씩 테스트 케이스를 작성하고 `run_test_suite`를 실행하세요.

- 수학 계산 (예: `'367 × 29는?'`)
- 날짜/달력 (예: `'2025년 3월 첫 번째 월요일은?'`)
- 단위 변환 (예: `'5 마일은 몇 킬로미터?'`)
- 논리 추론 (예: `'A가 B보다 크고 B가 C보다 크면 A와 C의 관계는?'`)
- 한국어 팩트 (예: `'대한민국 최장 강은?'`)

**목표**: Pass@1, 평균 응답 시간을 DataFrame으로 출력하고 Langfuse Dataset에 업로드

---

### Exercise 2: 모델 A vs B 성능 비교 실험

Exercise 1의 케이스에 5개를 추가(총 10개)하여 두 모델을 비교하세요.

- 실험 A: `claude-haiku-4-5`
- 실험 B: `claude-3-5-sonnet-20241022`

**목표**: Langfuse Experiment에 두 실험을 기록하고, 비교 DataFrame을 생성한 후  
"이 용도에는 어떤 모델이 적합한가?" 한 줄 결론을 작성하세요.


In [ ]:
# Exercise 1 — 직접 작성하세요
my_test_cases = [
    # {'id': 'MY-001', 'input': '...', 'expected': '...', 'category': 'math', 'level': 1},
    # 여기에 5개 추가
]

if my_test_cases:
    my_results = run_test_suite(agent_haiku, my_test_cases, 'my-exercise-1')
    df_my = pd.DataFrame(my_results)
    print(f"Pass@1: {df_my['score'].mean():.2%}")
    display(df_my[['id', 'status', 'score', 'latency']])
else:
    print('my_test_cases 리스트를 채워주세요!')